# Week 14 live coding: Paper package audit

Mục tiêu: dùng Python để kiểm tra paper package đã có gì, thiếu gì, và nên chọn Word + Zotero, Quarto hay Overleaf.

Core tuần này: audit package inventory, export missing-action table, tạo readiness figure, và viết mini paper package plan.

In [1]:
import hashlib
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve
try:
    import pandas as pd
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas==2.3.3", "matplotlib==3.9.4"])
    import pandas as pd
    import matplotlib.pyplot as plt
plt.rcParams["svg.hashsalt"] = "week14-paper-package"
print("pandas:", pd.__version__)
WEEK_PATH = Path("weeks/week-14-paper-package-overleaf")
WEEK_DIR = WEEK_PATH if WEEK_PATH.exists() else Path(".")
TABLE_DIR = WEEK_DIR / "outputs" / "tables"
FIG_DIR = WEEK_DIR / "outputs" / "figures"
DATA_DIR = WEEK_DIR / "data" / "raw"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_SHA = {
    "week14_paper_package_inventory.csv": "32d8e000dc6be913f58fc9bebd642df7fc93f2a447f737677d6963d08658af1e",
    "week14_reference_checklist.csv": "8c221a606662d94bcadb1321f8f21f157f461c844cabe51a9d000d4227fa6ad4",
}
REMOTE_BASE = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-14-paper-package-overleaf/data/raw"
for name, expected in EXPECTED_SHA.items():
    path = DATA_DIR / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"{REMOTE_BASE}/{name}", path)
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    print(name, actual)
    if actual != expected:
        raise ValueError(f"{name} SHA differs from the course snapshot. Stop and check the data file.")

pandas: 2.3.3
week14_paper_package_inventory.csv 32d8e000dc6be913f58fc9bebd642df7fc93f2a447f737677d6963d08658af1e
week14_reference_checklist.csv 8c221a606662d94bcadb1321f8f21f157f461c844cabe51a9d000d4227fa6ad4


## 1. Read the package inventory

Một row là một artifact trong paper package. Artifact có thể là question memo, data description, figure, reference file, Word draft, hoặc reproducibility note.

In [2]:
inventory = pd.read_csv(DATA_DIR / "week14_paper_package_inventory.csv")
print("Rows:", len(inventory))
print("Core items:", (inventory["required_core"] == "yes").sum())
print(inventory[["artifact_id", "paper_area", "artifact_name", "required_core", "status"]].head(10).to_string(index=False))

Rows: 16
Core items: 10
artifact_id paper_area                           artifact_name required_core status
        Q01    Framing        Research question and track memo           yes  ready
        D02       Data    Data description and unit definition           yes revise
        T03   Evidence                 Descriptive table model           yes revise
        C04       Data       Cleaning log / data decision note           yes  ready
        F05     Figure             Figure package with caption           yes  ready
        S06    Results     Statistics table / uncertainty note           yes  ready
        M07    Methods      TCSOL short-course method artifact            no  ready
        E08   Evidence                   Learner error summary            no  ready
       CV09   Evidence Chinese-Vietnamese contrastive examples            no  ready
        P10   Pedagogy            Pedagogical adaptation table            no  ready


## 2. Audit file paths and missing actions

`Path.exists()` kiểm tra file có thật trong folder hay chưa. Đây là checklist kỹ thuật; chất lượng paper vẫn cần người học tự đọc và sửa.

In [3]:
def path_exists(value):
    if pd.isna(value) or str(value).strip() == "":
        return False
    return Path(str(value)).exists()

inventory["file_exists"] = inventory["source_path"].apply(path_exists)
inventory["is_core"] = inventory["required_core"].eq("yes")
inventory["needs_action"] = inventory["is_core"] & ((inventory["status"] != "ready") | (~inventory["file_exists"]))
package_audit = inventory.copy()
package_audit.to_csv(TABLE_DIR / "week14_package_audit.csv", index=False)
missing_actions = package_audit[package_audit["needs_action"]][[
    "artifact_id", "paper_area", "artifact_name", "status", "file_exists", "learner_action", "notes"
]]
missing_actions.to_csv(TABLE_DIR / "week14_missing_actions.csv", index=False)
print("Core ready items:", int((package_audit["is_core"] & (package_audit["status"] == "ready") & package_audit["file_exists"]).sum()))
print("Core items needing action:", len(missing_actions))
print(missing_actions.to_string(index=False))

Core ready items: 5
Core items needing action: 5
artifact_id      paper_area                           artifact_name  status  file_exists                                                         learner_action                                         notes
        D02            Data    Data description and unit definition  revise         True             Write a 4-5 sentence data description for the chosen paper Use Week 02 as a template, not as final data.
        T03        Evidence                 Descriptive table model  revise         True        Pick the most relevant descriptive table for the selected track   A table without caption is not paper-ready.
      REF14      References Zotero reference library / bibliography missing        False Create Zotero collection and export .bib only if using Quarto/Overleaf Word + Zotero can avoid manual .bib at first.
    DRAFT14           Draft                        Mini paper draft missing        False                  Draft in Word first; 

## 3. Summarize readiness by paper area

Bảng này giúp học viên thấy vùng nào của paper đã ổn và vùng nào còn thiếu: references, draft, hoặc reproducibility note.

In [4]:
readiness = (
    package_audit
    .groupby(["paper_area", "status"], as_index=False)
    .size()
    .rename(columns={"size": "item_count"})
)
readiness.to_csv(TABLE_DIR / "week14_readiness_by_area.csv", index=False)
print(readiness.to_string(index=False))

     paper_area  status  item_count
           Data   ready           1
           Data  revise           1
          Draft missing           1
       Evidence   ready           4
       Evidence  revise           1
         Figure   ready           1
        Framing   ready           1
        Methods   ready           2
       Pedagogy   ready           1
     References missing           1
Reproducibility missing           1
        Results   ready           1


## 4. Choose the writing route

Tool choice là một research decision. Word + Zotero là default; Quarto và Overleaf chỉ dùng khi nhu cầu thật sự khớp.

In [5]:
tool_decision = pd.DataFrame([
    {
        "route": "Word + Zotero",
        "best_when": "beginner draft, familiar writing, standard applied-linguistics or education-policy paper",
        "required_files": "Word draft, Zotero collection, exported figures/tables",
        "risk": "manual file organization can become messy",
        "beginner_choice": "default",
    },
    {
        "route": "Quarto",
        "best_when": "the learner wants prose, code, citations, figures, and tables in one reproducible report",
        "required_files": "qmd file, notebook/code, figures/tables, references.bib",
        "risk": "adds Markdown/YAML/citation syntax",
        "beginner_choice": "optional",
    },
    {
        "route": "Overleaf",
        "best_when": "a LaTeX template, collaborator, journal, or university requires LaTeX",
        "required_files": "main.tex, references.bib, uploaded figures/tables",
        "risk": "formatting can distract from unfinished argument",
        "beginner_choice": "optional",
    },
])
tool_decision.to_csv(TABLE_DIR / "week14_tool_decision_table.csv", index=False)
print(tool_decision.to_string(index=False))

        route                                                                                best_when                                          required_files                                             risk beginner_choice
Word + Zotero beginner draft, familiar writing, standard applied-linguistics or education-policy paper  Word draft, Zotero collection, exported figures/tables        manual file organization can become messy         default
       Quarto the learner wants prose, code, citations, figures, and tables in one reproducible report qmd file, notebook/code, figures/tables, references.bib               adds Markdown/YAML/citation syntax        optional
     Overleaf                    a LaTeX template, collaborator, journal, or university requires LaTeX       main.tex, references.bib, uploaded figures/tables formatting can distract from unfinished argument        optional


## 5. Check reference workflow

Zotero status giúp học viên biết nguồn nào cần đưa vào Zotero, nguồn nào chỉ cần `.bib` nếu đi Quarto/Overleaf.

In [6]:
references = pd.read_csv(DATA_DIR / "week14_reference_checklist.csv")
reference_status = (
    references
    .groupby(["zotero_status", "citation_risk"], as_index=False)
    .size()
    .rename(columns={"size": "source_count"})
)
reference_status.to_csv(TABLE_DIR / "week14_reference_status.csv", index=False)
print(reference_status.to_string(index=False))
print()
print(references[["source_id", "source_title", "zotero_status", "bibtex_key"]].to_string(index=False))

zotero_status citation_risk  source_count
add_to_zotero           low             4
 optional_bib           low             1
 optional_bib        medium             3

source_id                                                   source_title zotero_status            bibtex_key
      R01                          Zotero. Using the Zotero Word Plugin. add_to_zotero    zotero-word-plugin
      R02                               Zotero. Creating Bibliographies. add_to_zotero zotero-bibliographies
      R03                                    Project TIER. Protocol 4.0. add_to_zotero project-tier-protocol
      R04                                             Quarto. Citations.  optional_bib      quarto-citations
      R05                                           Quarto. Manuscripts.  optional_bib    quarto-manuscripts
      R06                           Overleaf. Learn LaTeX in 30 minutes.  optional_bib  overleaf-learn-latex
      R07                        Overleaf Docs. Working with .bib fi

## 6. Export the readiness figure

Figure này không phải kết quả nghiên cứu. Nó là project-management figure để học viên thấy package đang ở trạng thái nào.

In [7]:
status_order = ["ready", "revise", "missing"]
pivot = readiness.pivot_table(index="paper_area", columns="status", values="item_count", fill_value=0)
for status in status_order:
    if status not in pivot.columns:
        pivot[status] = 0
pivot = pivot[status_order].sort_index()
colors = {"ready": "#1f7a4d", "revise": "#b45309", "missing": "#b8325f"}
fig, ax = plt.subplots(figsize=(9, 5.2))
left = None
for status in status_order:
    values = pivot[status]
    ax.barh(pivot.index, values, left=left, color=colors[status], label=status)
    left = values if left is None else left + values
ax.set_title("Week 14 paper package readiness")
ax.set_xlabel("Artifact count")
ax.set_ylabel("Paper area")
ax.legend(title="Status", loc="lower right")
ax.grid(axis="x", color="#d8e2f0")
fig.tight_layout()
fig.savefig(FIG_DIR / "week14_package_readiness.png", dpi=200)
fig.savefig(FIG_DIR / "week14_package_readiness.svg", metadata={"Date": "2026-06-04"})
plt.close(fig)
print("Figure exported:", FIG_DIR / "week14_package_readiness.png")

Figure exported: weeks/week-14-paper-package-overleaf/outputs/figures/week14_package_readiness.png


## 7. Paper-facing writing

Đoạn cuối cùng là package plan: học viên nói rõ paper sẽ dùng gì, thiếu gì, và viết bằng tool route nào.

In [8]:
ready_core = package_audit[package_audit["is_core"] & (package_audit["status"] == "ready") & package_audit["file_exists"]]
missing_names = "; ".join(missing_actions["artifact_name"].head(3).tolist())
paragraph = (
    "My mini paper package will use the course evidence as a small, transparent research folder rather than as one polished file. "
    "The current question will be narrowed to one track, with one explicit unit of analysis and one main table or figure. "
    f"The strongest ready core artifacts are {', '.join(ready_core['artifact_name'].head(4).tolist())}. "
    "I will draft the paper in Word with Zotero because this route keeps citation work familiar while the argument is still developing. "
    "Quarto or Overleaf will remain optional: Quarto is useful if I want code, figures, and citations in one reproducible report, while Overleaf is useful only if a LaTeX template or collaborator requires it. "
    f"Before submission, the package still needs action on: {missing_names}. "
    "The final folder should include the data inventory, runnable notebook, exported table or figure, caption, source note, Zotero references, and a short reproducibility note explaining how to find and rerun the work. "
    "This modest plan keeps formatting decisions tied to research evidence rather than tool novelty."
)
print(paragraph)
print()
print("Word count:", len(paragraph.split()))
repro_note = """## Reproducibility Note

- Data inventory: `weeks/week-14-paper-package-overleaf/data/raw/week14_paper_package_inventory.csv`
- Notebook: `weeks/week-14-paper-package-overleaf/live_coding.ipynb`
- Main outputs: `outputs/tables/week14_package_audit.csv` and `outputs/figures/week14_package_readiness.png`
- Reference workflow: Word + Zotero by default; export `.bib` only for Quarto/Overleaf.
- Rerun note: open the notebook from top to bottom after checking that source paths still exist.
"""
submission_text = f"""# Week 14 Submission Plan Model

## Mini Paper Package Plan

{paragraph}

{repro_note}
"""
(TABLE_DIR / "week14_submission_plan.md").write_text(submission_text, encoding="utf-8")
print("Submission plan exported:", TABLE_DIR / "week14_submission_plan.md")

My mini paper package will use the course evidence as a small, transparent research folder rather than as one polished file. The current question will be narrowed to one track, with one explicit unit of analysis and one main table or figure. The strongest ready core artifacts are Research question and track memo, Cleaning log / data decision note, Figure package with caption, Statistics table / uncertainty note. I will draft the paper in Word with Zotero because this route keeps citation work familiar while the argument is still developing. Quarto or Overleaf will remain optional: Quarto is useful if I want code, figures, and citations in one reproducible report, while Overleaf is useful only if a LaTeX template or collaborator requires it. Before submission, the package still needs action on: Data description and unit definition; Descriptive table model; Zotero reference library / bibliography. The final folder should include the data inventory, runnable notebook, exported table or fi